# M2SVid Lightning tiny demo

Refinement-only Lightning notebook. Assumes the M2SVid source tree is pasted at `/teamspace/studios/this_studio`.

## 0. Check paths and GPU

In [ ]:
from pathlib import Path
import os, time, shutil, subprocess, json
STUDIO_ROOT = Path('/teamspace/studios/this_studio')
CODE_ROOT = STUDIO_ROOT
DATA_ROOT = STUDIO_ROOT / 'm2svid_demo'
CKPT_DIR = DATA_ROOT / 'ckpts'
OUTPUT_ARCHIVE = DATA_ROOT / 'outputs'
for p in [DATA_ROOT, CKPT_DIR, OUTPUT_ARCHIVE]: p.mkdir(parents=True, exist_ok=True)
os.chdir(CODE_ROOT)
print('CODE_ROOT=', CODE_ROOT)
print('README exists:', (CODE_ROOT/'README.md').exists())
print('demo exists:', (CODE_ROOT/'demo').exists())
!pwd
!nvidia-smi
!find demo -maxdepth 3 -type f | sort | head -50

## 0.5. Install/check system video tools


In [ ]:
# System-level video tools required by ffmpeg-python
# Lightning allows sudo apt-get in Studio. Safe to rerun.
!which ffmpeg || (sudo apt-get update -qq && sudo apt-get install -y ffmpeg)
!which ffprobe || true
!ffmpeg -version | head -1


## 1. View official precomputed demo outputs

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path

def show_video(path, width=720):
    path = Path(path)
    assert path.exists(), f'Missing {path}'
    data = 'data:video/mp4;base64,' + b64encode(path.read_bytes()).decode()
    display(HTML(f'<video width="{width}" controls loop><source src="{data}" type="video/mp4"></video><p><code>{path}</code></p>'))

print('Input'); show_video(CODE_ROOT/'demo/input.mp4', 520)
print('Warped right'); show_video(CODE_ROOT/'demo/reprojected/input_reprojected.mp4', 520)
print('Mask'); show_video(CODE_ROOT/'demo/reprojected/input_reprojected_mask.mp4', 520)
print('Official M2SVid SBS'); show_video(CODE_ROOT/'demo/m2svid/input_sbs.mp4', 720)

## 2. Download/link checkpoints to Lightning persistent storage

In [ ]:
os.chdir(CODE_ROOT)
!python -m pip -q install gdown
if not (CKPT_DIR/'m2svid_weights.pt').exists():
    !wget -c -O {CKPT_DIR/'m2svid_weights.pt'} https://storage.googleapis.com/gresearch/m2svid/m2svid_weights.pt
else:
    print('M2SVid weight exists:', CKPT_DIR/'m2svid_weights.pt')
hi3d_zip = CKPT_DIR/'hi3d_ckpts.zip'
if not hi3d_zip.exists() and not (CKPT_DIR/'open_clip_pytorch_model.bin').exists() and not (CODE_ROOT/'ckpts/open_clip_pytorch_model.bin').exists():
    !python -m gdown 1j_NEG2CPhFeRetYziWK6Qe62R5h7lG_V -O {hi3d_zip}
else:
    print('Hi3D/OpenCLIP already present or zip exists')
!mkdir -p {CODE_ROOT/'ckpts'}
if hi3d_zip.exists() and not (CODE_ROOT/'ckpts/open_clip_pytorch_model.bin').exists():
    !unzip -n {hi3d_zip} -d {CODE_ROOT}
if (CKPT_DIR/'open_clip_pytorch_model.bin').exists():
    !ln -sf {CKPT_DIR/'open_clip_pytorch_model.bin'} {CODE_ROOT/'ckpts/open_clip_pytorch_model.bin'}
!ln -sf {CKPT_DIR/'m2svid_weights.pt'} {CODE_ROOT/'ckpts/m2svid_weights.pt'}
!find ckpts -maxdepth 2 -type f -o -type l | sort | head -80

## 3. Install minimal deps

In [ ]:
os.chdir(CODE_ROOT)
!python -m pip -q install omegaconf einops pytorch-lightning==1.5.0 kornia==0.6.9 open-clip-torch==2.29.0 opencv-python-headless==4.10.0.84 omegaconf==2.3.0 einops==0.8.0 pytorch-lightning==1.5.0 kornia==0.6.9 open-clip-torch==2.29.0 pytorch-msssim==1.0.0 transformers diffusers accelerate tokenizers sentencepiece ftfy ffmpeg-python av imageio imageio-ffmpeg safetensors scipy scikit-image pillow tqdm matplotlib pandas numpy==1.26.4 xformers || true

# Check from the current notebook kernel instead of using a shell heredoc.
# In IPython, `!python - <<'PY' ... PY` can be parsed incorrectly line-by-line.
import torch, torchvision
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())
print('torchvision', torchvision.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 4. Run tiny inference

In [ ]:
os.chdir(CODE_ROOT)
out = CODE_ROOT/'outputs/m2svid_lightning_tiny'
if out.exists(): shutil.rmtree(out)
out.mkdir(parents=True, exist_ok=True)
start=time.time()
cmd = '''PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True PYTHONPATH="./:./third_party/Hi3D-Official/:./third_party/pytorch-msssim/:${PYTHONPATH}" python inpaint_and_refine.py \
--mask_antialias 0 \
--model_config configs/m2svid.yaml \
--ckpt ckpts/m2svid_weights.pt \
--video_path demo_lowmem/input.mp4 \
--reprojected_path demo_lowmem/reprojected/input_reprojected.mp4 \
--reprojected_mask_path demo_lowmem/reprojected/input_reprojected_mask.mp4 \
--output_folder outputs/m2svid_lightning_tiny'''
print(cmd)
ret=subprocess.run(cmd, shell=True)
print('return code:', ret.returncode, 'elapsed:', round(time.time()-start,1))
!find outputs/m2svid_lightning_tiny -type f -print
!nvidia-smi

## 5. View and archive

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
from pathlib import Path

def show_video(path, width=720):
    path = Path(path)
    assert path.exists(), f'Missing {path}'
    data = 'data:video/mp4;base64,' + b64encode(path.read_bytes()).decode()
    display(HTML(f'<video width="{width}" controls loop><source src="{data}" type="video/mp4"></video><p><code>{path}</code></p>'))

out_dir = CODE_ROOT/'outputs/m2svid_lightning_tiny'
for p in sorted(out_dir.glob('*.mp4')): print(p, round(p.stat().st_size/1e6,2),'MB')
if (out_dir/'input_sbs.mp4').exists(): show_video(out_dir/'input_sbs.mp4', 720)
elif (out_dir/'input_generated.mp4').exists(): show_video(out_dir/'input_generated.mp4', 520)
archive = OUTPUT_ARCHIVE / time.strftime('tiny_%Y%m%d_%H%M%S')
archive.mkdir(parents=True, exist_ok=True)
for p in out_dir.glob('*'): shutil.copy2(p, archive/p.name)
print('Archived to:', archive)